# Issue #1092 — Exploring Existing PyIceberg Infrastructure for Compaction

This notebook explores the existing building blocks in PyIceberg that will be used
to implement `rewrite_data_files`. The goal is to understand what exists, see it in
action, and identify exactly what needs to be built.

**Run with**: `make notebook` from the project root, or:
```bash
export PATH="$HOME/.local/bin:$PATH"
uv run --extra pyarrow --extra pandas --extra sql-sqlite --group notebook jupyter lab --notebook-dir=notebooks
```

## 1. Setup — Create a test table with many small files

We use an in-memory SQLite catalog so nothing persists outside `/tmp/`.

In [1]:
import pyarrow as pa
import os
import shutil

from pyiceberg.catalog.sql import SqlCatalog
from pyiceberg.schema import Schema
from pyiceberg.types import NestedField, StringType, LongType, DateType
from pyiceberg.partitioning import PartitionSpec, PartitionField
from pyiceberg.transforms import IdentityTransform

# Clean up from any previous run
WAREHOUSE = "/tmp/iceberg_compaction_exploration"
if os.path.exists(WAREHOUSE):
    shutil.rmtree(WAREHOUSE)

catalog = SqlCatalog(
    "test_catalog",
    **{
        "uri": "sqlite:////tmp/iceberg_compaction_test.db",
        "warehouse": f"file://{WAREHOUSE}",
    }
)

# Create namespace
try:
    catalog.create_namespace("test_db")
except Exception:
    pass  # already exists

print(f"Catalog ready. Warehouse: {WAREHOUSE}")

Catalog ready. Warehouse: /tmp/iceberg_compaction_exploration


In [2]:
# Create a partitioned table
schema = Schema(
    NestedField(1, "id", LongType(), required=False),
    NestedField(2, "name", StringType(), required=False),
    NestedField(3, "category", StringType(), required=False),
    NestedField(4, "value", LongType(), required=False),
)

partition_spec = PartitionSpec(
    PartitionField(source_id=3, field_id=1000, transform=IdentityTransform(), name="category")
)

try:
    catalog.drop_table("test_db.events")
except Exception:
    pass

table = catalog.create_table(
    "test_db.events",
    schema=schema,
    partition_spec=partition_spec,
)

print(f"Table created: {table.name()}")
print(f"Schema: {table.schema()}")
print(f"Partition spec: {table.spec()}")
print(f"Properties: {table.properties}")

Table created: ('test_db', 'events')
Schema: table {
  1: id: optional long
  2: name: optional string
  3: category: optional string
  4: value: optional long
}
Partition spec: [
  1000: category: identity(3)
]
Properties: {}


In [3]:
# Append many small batches to simulate the "small files" problem
# Each append creates a separate data file
import random

categories = ["electronics", "clothing", "food"]

for i in range(15):
    category = categories[i % len(categories)]
    df = pa.table({
        "id": [i * 100 + j for j in range(50)],
        "name": [f"item_{i}_{j}" for j in range(50)],
        "category": [category] * 50,
        "value": [random.randint(1, 1000) for _ in range(50)],
    })
    table.append(df)

print(f"Appended 15 batches (5 per partition, 50 rows each = 750 total rows)")

Appended 15 batches (5 per partition, 50 rows each = 750 total rows)


## 2. Inspect current state — many small files per partition

In [4]:
# This is the EXISTING infrastructure for Requirement 1:
# table.scan(row_filter=...).plan_files() — finds data files matching a filter

from pyiceberg.expressions import AlwaysTrue, EqualTo

# Scan ALL files (no filter)
all_tasks = list(table.scan(row_filter=AlwaysTrue()).plan_files())

print(f"Total data files: {len(all_tasks)}")
print(f"Total rows: {sum(t.file.record_count for t in all_tasks)}")
print()

for task in all_tasks:
    f = task.file
    print(f"  {os.path.basename(f.file_path):40s}  "
          f"partition={f.partition}  "
          f"size={f.file_size_in_bytes:>8,} bytes  "
          f"rows={f.record_count}")

Total data files: 15
Total rows: 750

  00000-0-8fb81746-0a23-401d-b2f6-a3890b43453a.parquet  partition=Record[food]  size=   2,190 bytes  rows=50
  00000-0-0ef8bb2d-53f9-43f4-b69d-33b7abd7697e.parquet  partition=Record[clothing]  size=   2,204 bytes  rows=50
  00000-0-b6213c62-d88e-48b6-ac0b-406e885ebcb4.parquet  partition=Record[electronics]  size=   2,221 bytes  rows=50
  00000-0-2e57fe15-b514-451e-b9f7-221d33b007a9.parquet  partition=Record[food]  size=   2,186 bytes  rows=50
  00000-0-87f7cb07-2396-4ff1-a3a6-8708155f9a6d.parquet  partition=Record[clothing]  size=   2,204 bytes  rows=50
  00000-0-872c3267-edd9-446a-a992-38eb0c67153f.parquet  partition=Record[electronics]  size=   2,208 bytes  rows=50
  00000-0-d5a740cc-cd5f-4d71-8533-66dc4db3fd0a.parquet  partition=Record[food]  size=   2,179 bytes  rows=50
  00000-0-2aff41dd-e5d3-4542-a8aa-515827846544.parquet  partition=Record[clothing]  size=   2,190 bytes  rows=50
  00000-0-6e058d1f-7d1c-4dcc-afe7-3d2609a4ab5f.parquet  partitio

In [5]:
# Scan with FILTER — only files matching category='electronics'
# This is Requirement 1 in action

electronics_tasks = list(
    table.scan(row_filter=EqualTo("category", "electronics")).plan_files()
)

print(f"Files matching category='electronics': {len(electronics_tasks)}")
for task in electronics_tasks:
    f = task.file
    print(f"  {os.path.basename(f.file_path):40s}  "
          f"partition={f.partition}  "
          f"size={f.file_size_in_bytes:>8,} bytes")

Files matching category='electronics': 5
  00000-0-b6213c62-d88e-48b6-ac0b-406e885ebcb4.parquet  partition=Record[electronics]  size=   2,221 bytes
  00000-0-872c3267-edd9-446a-a992-38eb0c67153f.parquet  partition=Record[electronics]  size=   2,208 bytes
  00000-0-6e058d1f-7d1c-4dcc-afe7-3d2609a4ab5f.parquet  partition=Record[electronics]  size=   2,207 bytes
  00000-0-568f46bb-7b07-431a-9c84-d8d99ba295b9.parquet  partition=Record[electronics]  size=   2,216 bytes
  00000-0-d628804c-9e3b-47d9-b2c2-7d6be47a42a6.parquet  partition=Record[electronics]  size=   2,163 bytes


## 3. Explore the building blocks for Requirement 2

### 3a. Group by partition

In [6]:
# GROUP BY PARTITION — this is what we need to build (~5 lines)
from collections import defaultdict

files_by_partition = defaultdict(list)
for task in all_tasks:
    # task.file.partition is a Record with the partition values
    partition_key = str(task.file.partition)  # use string repr as dict key
    files_by_partition[partition_key].append(task)

print("Files grouped by partition:")
for partition, tasks in files_by_partition.items():
    total_size = sum(t.file.file_size_in_bytes for t in tasks)
    print(f"  {partition}: {len(tasks)} files, {total_size:,} bytes total")

Files grouped by partition:
  Record[food]: 5 files, 10,868 bytes total
  Record[clothing]: 5 files, 10,964 bytes total
  Record[electronics]: 5 files, 11,015 bytes total


### 3b. ListPacker — the existing bin-packing algorithm

In [7]:
# The ListPacker ALREADY EXISTS in pyiceberg
from pyiceberg.utils.bin_packing import ListPacker

# Demo: pack file sizes [100, 200, 300, 150, 250, 50] into bins of target_weight=500
demo_items = [100, 200, 300, 150, 250, 50]
packer = ListPacker(target_weight=500, lookback=1, largest_bin_first=False)
bins = packer.pack(demo_items, weight_func=lambda x: x)

print("Demo: Packing [100, 200, 300, 150, 250, 50] into bins of max 500:")
for i, b in enumerate(bins):
    print(f"  Bin {i}: {b} (total={sum(b)})")

print()

# Now pack our actual file tasks
MAX_GROUP_SIZE = 100 * 1024 * 1024 * 1024  # 100GB (same as Java default)
packer = ListPacker(target_weight=MAX_GROUP_SIZE, lookback=1, largest_bin_first=False)

for partition, tasks in files_by_partition.items():
    groups = packer.pack(tasks, weight_func=lambda t: t.file.file_size_in_bytes)
    print(f"Partition {partition}: {len(tasks)} files → {len(groups)} groups")
    for i, group in enumerate(groups):
        total = sum(t.file.file_size_in_bytes for t in group)
        print(f"  Group {i}: {len(group)} files, {total:,} bytes")

Demo: Packing [100, 200, 300, 150, 250, 50] into bins of max 500:
  Bin 0: [100, 200] (total=300)
  Bin 1: [300, 150] (total=450)
  Bin 2: [250, 50] (total=300)

Partition Record[food]: 5 files → 1 groups
  Group 0: 5 files, 10,868 bytes
Partition Record[clothing]: 5 files → 1 groups
  Group 0: 5 files, 10,964 bytes
Partition Record[electronics]: 5 files → 1 groups
  Group 0: 5 files, 11,015 bytes


### 3c. The writer's bin-packing constraints

"Using the same bin-packing constraints of the writer" means reusing the existing
`_dataframe_to_data_files()` pipeline which internally uses `bin_pack_arrow_table()`
with `write.target-file-size-bytes`.

In [8]:
# Inspect the writer's target file size
from pyiceberg.table import TableProperties

target_file_size = int(table.properties.get(
    TableProperties.WRITE_TARGET_FILE_SIZE_BYTES,
    TableProperties.WRITE_TARGET_FILE_SIZE_BYTES_DEFAULT,
))

min_file_size = int(target_file_size * 0.75)  # Java: MIN_FILE_SIZE_DEFAULT_RATIO
max_file_size = int(target_file_size * 1.80)  # Java: MAX_FILE_SIZE_DEFAULT_RATIO

print(f"Writer's target file size: {target_file_size:>15,} bytes ({target_file_size / 1024**2:.0f} MB)")
print(f"Min file size (75%):       {min_file_size:>15,} bytes ({min_file_size / 1024**2:.0f} MB)")
print(f"Max file size (180%):      {max_file_size:>15,} bytes ({max_file_size / 1024**2:.0f} MB)")

print()
print("File sizes vs thresholds:")
for task in all_tasks:
    size = task.file.file_size_in_bytes
    status = "✅ optimal" if min_file_size <= size <= max_file_size else "🔴 needs rewrite"
    print(f"  {os.path.basename(task.file.file_path):40s}  {size:>8,} bytes  {status}")

Writer's target file size:     536,870,912 bytes (512 MB)
Min file size (75%):           402,653,184 bytes (384 MB)
Max file size (180%):          966,367,641 bytes (922 MB)

File sizes vs thresholds:
  00000-0-8fb81746-0a23-401d-b2f6-a3890b43453a.parquet     2,190 bytes  🔴 needs rewrite
  00000-0-0ef8bb2d-53f9-43f4-b69d-33b7abd7697e.parquet     2,204 bytes  🔴 needs rewrite
  00000-0-b6213c62-d88e-48b6-ac0b-406e885ebcb4.parquet     2,221 bytes  🔴 needs rewrite
  00000-0-2e57fe15-b514-451e-b9f7-221d33b007a9.parquet     2,186 bytes  🔴 needs rewrite
  00000-0-87f7cb07-2396-4ff1-a3a6-8708155f9a6d.parquet     2,204 bytes  🔴 needs rewrite
  00000-0-872c3267-edd9-446a-a992-38eb0c67153f.parquet     2,208 bytes  🔴 needs rewrite
  00000-0-d5a740cc-cd5f-4d71-8533-66dc4db3fd0a.parquet     2,179 bytes  🔴 needs rewrite
  00000-0-2aff41dd-e5d3-4542-a8aa-515827846544.parquet     2,190 bytes  🔴 needs rewrite
  00000-0-6e058d1f-7d1c-4dcc-afe7-3d2609a4ab5f.parquet     2,207 bytes  🔴 needs rewrite
  00000

### 3d. Reading files with ArrowScan (existing)

In [9]:
# ArrowScan.to_table() can read a list of FileScanTasks into an Arrow table
# This is what the rewriter will use to READ file groups

# Pick one partition's files to demo
partition_key = list(files_by_partition.keys())[0]
partition_tasks = files_by_partition[partition_key]

# Read them all into one Arrow table using the existing scan
arrow_table = table.scan(
    row_filter=EqualTo("category", partition_key.split("'")[1] if "'" in partition_key else "electronics")
).to_arrow()

print(f"Read {len(partition_tasks)} files into single Arrow table:")
print(f"  Rows: {arrow_table.num_rows}")
print(f"  Columns: {arrow_table.column_names}")
print(f"  Memory: {arrow_table.nbytes:,} bytes")
print()
print(arrow_table.to_pandas().head(10))

Read 5 files into single Arrow table:
  Rows: 250
  Columns: ['id', 'name', 'category', 'value']
  Memory: 11,070 bytes

     id       name     category  value
0  1200  item_12_0  electronics    998
1  1201  item_12_1  electronics    583
2  1202  item_12_2  electronics    175
3  1203  item_12_3  electronics    261
4  1204  item_12_4  electronics    251
5  1205  item_12_5  electronics    347
6  1206  item_12_6  electronics    609
7  1207  item_12_7  electronics    108
8  1208  item_12_8  electronics     86
9  1209  item_12_9  electronics    528


### 3e. MaintenanceTable — where rewrite_data_files will live

In [10]:
# The MaintenanceTable class already exists with expire_snapshots()
# rewrite_data_files() will be added here

maintenance = table.maintenance
print(f"MaintenanceTable class: {type(maintenance)}")
print(f"Available methods: {[m for m in dir(maintenance) if not m.startswith('_')]}")
print()
print("Current snapshots:")
for snap in table.metadata.snapshots:
    print(f"  Snapshot {snap.snapshot_id}: {snap.summary}")

MaintenanceTable class: <class 'pyiceberg.table.maintenance.MaintenanceTable'>
Available methods: ['expire_snapshots', 'tbl']

Current snapshots:
  Snapshot 6457227091833542474: operation=Operation.APPEND
  Snapshot 7168899509377265900: operation=Operation.APPEND
  Snapshot 8620729704389039574: operation=Operation.APPEND
  Snapshot 7356709084472223558: operation=Operation.APPEND
  Snapshot 6419802005031762909: operation=Operation.APPEND
  Snapshot 5199072272643781002: operation=Operation.APPEND
  Snapshot 2401396837408880549: operation=Operation.APPEND
  Snapshot 5839589466410590286: operation=Operation.APPEND
  Snapshot 862033521299312338: operation=Operation.APPEND
  Snapshot 5030968198705008929: operation=Operation.APPEND
  Snapshot 4075109043274892389: operation=Operation.APPEND
  Snapshot 9180190020035318795: operation=Operation.APPEND
  Snapshot 7092494148821833341: operation=Operation.APPEND
  Snapshot 3770353112811587841: operation=Operation.APPEND
  Snapshot 552950481238076834

### 3f. _OverwriteFiles — the commit mechanism

In [11]:
# Inspect the _OverwriteFiles class that handles atomic file replacement
from pyiceberg.table.update.snapshot import _OverwriteFiles

print(f"_OverwriteFiles class: {_OverwriteFiles}")
print(f"Methods: {[m for m in dir(_OverwriteFiles) if not m.startswith('__')]}")
print()
print("This handles the atomic delete-old + add-new snapshot update.")
print("It's already used by table.overwrite() for existing data overwrites.")

_OverwriteFiles class: <class 'pyiceberg.table.update.snapshot._OverwriteFiles'>
Methods: ['_abc_impl', '_build_delete_files_partition_predicate', '_build_manifest_evaluator', '_build_partition_projection', '_calculate_added_rows', '_commit', '_deleted_entries', '_existing_manifests', '_manifests', '_process_manifests', '_summary', '_validate_target_branch', 'append_data_file', 'commit', 'delete_by_predicate', 'delete_data_file', 'fetch_manifest_entry', 'new_manifest_output', 'new_manifest_writer', 'partition_filters', 'schema', 'snapshot_id', 'spec']

This handles the atomic delete-old + add-new snapshot update.
It's already used by table.overwrite() for existing data overwrites.


## 4. Summary: What exists vs what's needed

| Component | Status | Location |
|---|---|---|
| `table.scan(filter).plan_files()` | ✅ EXISTS | `pyiceberg/table/__init__.py` |
| Expression types (EqualTo, etc.) | ✅ EXISTS | `pyiceberg/expressions/` |
| `ListPacker` bin-packing | ✅ EXISTS | `pyiceberg/utils/bin_packing.py` |
| `ArrowScan.to_table()` read | ✅ EXISTS | `pyiceberg/io/pyarrow.py` |
| `_dataframe_to_data_files()` write | ✅ EXISTS | `pyiceberg/io/pyarrow.py` |
| `_OverwriteFiles` commit | ✅ EXISTS | `pyiceberg/table/update/snapshot.py` |
| `MaintenanceTable` class | ✅ EXISTS | `pyiceberg/table/maintenance.py` |
| Group by partition | 🆕 NEW | ~5 lines |
| Filter files by size | 🆕 NEW | ~10 lines |
| Filter groups by min_input_files | 🆕 NEW | ~5 lines |
| Orchestration | 🆕 NEW | ~50 lines |